# Training BPNN potential using n2p2  with AiiDA workflows

This notebook is a demonstration of MLP fitting using aiida-n2p2 plugin.

We are going to train an n2p2 MLP for Aluminium using the AiiDA-n2p2 plugin:

1. **`nnp-scaling`** to generate symmetry functions in tabular format
2. **`nnp-train`** to train the machine learning potential (MLP)
3. **`AiiDA-LAMMPS`** plugin to validate our MLP by obtaining the radial distribution function (RDF)

In this demonstration, all calculations will be run on the local computer (MAC OS).

## DFT traning data set for liquid Aluminium

Training set configuration:

1. 1029 AIMD snapshots
2. 256 atoms in a box of 12.5 Angstroms
3. Temperature 8000K


In [ ]:
from ase.visualize import view
from ase.io import read, write
import nglview as nv
atoms=read("../1.Data/input.xyz",index=":")
view(atoms, viewer='ngl')


## Workflow: Import modules


In [ ]:
from pathlib import Path

from aiida.engine import run
from aiida import orm
from aiida.orm import Int, SinglefileData, Dict, Code, load_node
from aiida.plugins import WorkflowFactory
from aiida.manage.configuration import load_profile

import warnings
warnings.filterwarnings('ignore')

%load_ext aiida


## Workflow: AiiDA  settings 

In [ ]:
load_profile()      # This will load the aiida profile
INPUT_DIR = Path().resolve()  # Path of the script

# Define computer
computer = orm.load_computer('local')

#Define codes
scaleCode = orm.InstalledCode(
        label='n2p2', computer=computer, filepath_executable='nnp-scaling', default_calc_job_plugin='n2p2.scale'
    )
trainCode = orm.InstalledCode(
        label='n2p2', computer=computer, filepath_executable='nnp-train', default_calc_job_plugin='n2p2.train'
    )


## Workflow: Setup inputs

In [4]:

# Parameters for n2p2 training 
nbin = orm.Int(100)
inputData = orm.SinglefileData(file=INPUT_DIR / 'input.data')
inputNN = orm.SinglefileData(file=INPUT_DIR / 'input.nn')
atomicNumber=orm.Int(13)

#LAMMPS validation input
lammpsCode=orm.load_code("lammps@local")
lammpsScript=orm.SinglefileData(file=INPUT_DIR / 'in.lmp')
lammpsData=orm.SinglefileData(file=INPUT_DIR / '222_IN.data')

#Define resources
scale_metadata = Dict(dict={"options": {"resources": {"num_machines": 1},"withmpi":True}})
train_metadata = Dict(dict={"options": {"resources": {"num_machines": 1}}})
validate_metadata = Dict(dict={"options": {"resources": {"num_machines": 1},"withmpi":True}})



## Run the Workflow

In [ ]:

train_n2p2=WorkflowFactory('n2p2.make_potential')

# Run the WorkChain
result = run(
    train_n2p2,
    code=scaleCode,
    nbin=nbin,
    inputData=inputData,
    inputNN=inputNN,
    trainCode=trainCode,
    atomicNumber=atomicNumber,
    lammpsCode=lammpsCode,
    lammpsScript=lammpsScript,
    lammpsData=lammpsData,
    n2p2= {
        "scale":
            {
                "metadata": scale_metadata
                },

        "train":
        {
            "metadata": train_metadata
            },
    "validate":
    {
        "metadata":validate_metadata
        }
    }
    )




## Check the status of all the calculations


In [ ]:
verdi process status 1804

## Visualize the LAMMPS Trajectories


In [ ]:
lammps_atoms = read("dump_equil.lammpstrj", index=":")

# Assign  the chemical symbol for all atoms in each frame
for struct in lammps_atoms:
    struct.set_chemical_symbols(['Al'] * len(struct))
    
view(lammps_atoms, viewer='ngl')


# Comparison of AIMD and MLIP MD

In [ ]:
from ase.io import read
from ase.visualize import view
from ipywidgets import HBox, VBox, Layout, HTML
from IPython.display import display

# Load atoms

# Create the NGL views
view1 = view(atoms, viewer='ngl')
view2 = view(lammps_atoms, viewer='ngl')

# Create labels for titles
title1 = HTML("<h3>Visualization 1: AIMD 256 Atoms</h3>")  
title2 = HTML("<h3>Visualization 2: MLIP MD 2048 Atoms</h3>")

# Combine each title with its respective view in a VBox
view1_with_title = VBox([title1, view1])
view2_with_title = VBox([title2, view2])

# Combine both titled views side by side in an HBox
HBox([view1_with_title, view2_with_title])

## Plot the Radial Distribution Function


In [ ]:
!export MPLBACKEND=Agg&& python3 rdf_comparison.py && open output/rdf_plot.png

## Plot the provenance graph

In [ ]:
verdi node graph generate 1804

In [ ]:
!open 1804.dot.pdf
